# Transformer Experiments
COMP6242 RNN's Revenge — GPT + RoPE + RMSNorm + Flash Attention

Runs 9 experiments: 3 tasks × 3 lengths.

| Task | Lengths | Steps | Script |
|------|---------|-------|--------|
| Shakespeare | 256 / 1024 / 2048 | 5K | `train.py` |
| Copy | short / medium / long | 5K | `train_synthetic.py` |
| Induction | short / medium / long | 20K (lr_decay 50K) | `train_synthetic.py` |

Protocol: dropout 0.05, AdamW (0.9/0.95), wd=0.1, lr 3e-4→3e-5, seed 42.

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'bf16: {torch.cuda.is_bf16_supported()}')

In [ ]:
%cd /content/DL-RNNs-Revenge/transformer_project

## Data Generation

In [ ]:
!python data/tinyshakespeare/prepare.py
!python generate_longrange_copy.py
!python generate_induction.py

## Shakespeare × 3

In [ ]:
!python train.py config/tshake_256.py
!python train.py config/tshake_1024.py
!python train.py config/tshake_2048.py

## Long-range Copy × 3

In [ ]:
!python train_synthetic.py config/copy_short.py
!python train_synthetic.py config/copy_medium.py
!python train_synthetic.py config/copy_long.py

## Induction × 3 (20K steps, lr_decay 50K)

In [ ]:
!python train_synthetic.py config/induction_short.py
!python train_synthetic.py config/induction_medium.py
!python train_synthetic.py config/induction_long.py

## Collect Results

In [ ]:
import json, glob

print(f"{'Run':<30} {'Val PPL':>10} {'Disc':>12} {'Acc':>8}")
print('-' * 65)
for p in sorted(glob.glob('out/*/summary.json')):
    with open(p) as f:
        s = json.load(f)
    name = s['run_name']
    ppl = f"{s['best_val_ppl']:.4f}"
    if 'best_recall_ppl' in s:
        disc = f"{s['best_recall_ppl']:.4f}"
        acc = '-'
    elif 'best_induction5_accuracy' in s:
        disc = f"{s['best_pattern_ppl']:.4f}"
        acc = f"{s['best_induction5_accuracy']:.4f}"
    else:
        disc = '-'
        acc = '-'
    print(f"{name:<30} {ppl:>10} {disc:>12} {acc:>8}")